### **Submissão 1B — Modelo PyTorch**

**Grupo 1 · MIA · Aprendizagem Profunda**

Modelo: DNN (PyTorch) com Dropout + Weight Decay  
Output: `subm1-g1-MIA-B.csv`

In [1]:
import numpy as np
import pandas as pd
import pickle
import sys, os
import torch
import torch.nn as nn

sys.path.append(os.path.abspath('../src'))
from utils import transform_new_texts

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

print("\n1. A carregar modelo PyTorch...")
with open('../models/pytorch.pkl', 'rb') as f:
    meta = pickle.load(f)

transformers = meta['transformers']
class_names = meta['class_names']
model_class = meta['model_class']
input_size = meta['input_size']
num_classes = meta['num_classes']

print(f"   Modelo: {model_class}")
print(f"   Classes: {list(class_names)}")

Dispositivo: cuda

1. A carregar modelo PyTorch...
   Modelo: DNNBaseline
   Classes: [np.str_('Anthropic'), np.str_('Google'), np.str_('Human'), np.str_('Meta'), np.str_('OpenAI')]


In [2]:
print("2. A carregar dataset de submissão...")
df = pd.read_csv('../data/subm1.csv', sep=';')
df.columns = df.columns.str.strip().str.lower()

textos = df['text'].tolist()
ids = df['id'].tolist()
print(f"   {len(textos)} textos carregados")

2. A carregar dataset de submissão...
   150 textos carregados


In [3]:
print("3. A extrair features...")
X = transform_new_texts(textos, transformers)
print(f"   {X.shape[0]} textos → {X.shape[1]} features")

3. A extrair features...
   150 textos → 2013 features


In [4]:
print("4. A classificar...")

# Definir a arquitetura (tem de coincidir com o treino)
class DNNBaseline(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, num_classes))
    def forward(self, x): return self.net(x)

class DNNBatchNorm(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x): return self.net(x)

class DNNWide(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x): return self.net(x)

model_map = {'DNNBaseline': DNNBaseline, 'DNNBatchNorm': DNNBatchNorm, 'DNNWide': DNNWide}

model = model_map[model_class](input_size, num_classes).to(device)
model.load_state_dict(torch.load('../models/pytorch.pth', map_location=device, weights_only=True))
model.eval()

X_t = torch.FloatTensor(X).to(device)
with torch.no_grad():
    _, y_pred = torch.max(model(X_t), 1)
    y_pred = y_pred.cpu().numpy()

labels_pred = [class_names[i] for i in y_pred]
print(f"   ✅ {len(labels_pred)} previsões feitas")

4. A classificar...
   ✅ 150 previsões feitas


In [5]:
print("5. A exportar CSV...")

df_out = pd.DataFrame({
    'ID': ids,
    'Text': textos,
    'Labels': labels_pred
})

output_path = '../Subm1/subm1-g1-MIA-B.csv'
os.makedirs('../Subm1', exist_ok=True)
df_out.to_csv(output_path, sep=';', index=False, encoding='utf-8')

print(f"✅ Ficheiro guardado: {output_path}")
print(f"\nDistribuição das previsões:")
print(df_out['Labels'].value_counts().to_string())
print(f"\nPrimeiras 10 linhas:")
print(df_out.head(10).to_string(index=False))

5. A exportar CSV...
✅ Ficheiro guardado: ../Subm1/subm1-g1-MIA-B.csv

Distribuição das previsões:
Labels
Human        66
Anthropic    41
Google       24
Meta         11
OpenAI        8

Primeiras 10 linhas:
   ID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   